# TD1 — Web Mining Pipeline (Project)

**Domain:** Sustainable / Renewable Energy  

**Outputs (in `td1/outputs/`):**
- `crawler_output.jsonl`
- `extracted_knowledge.csv`
- `relation_candidates.csv`
- `crawl_summary.csv`

In [ ]:
# ── 0. Dependency bootstrap ────────────────────────────────────────────────
import importlib, subprocess, sys

def ensure(import_name, pip_name=None):
    pip_name = pip_name or import_name
    try:
        importlib.import_module(import_name)
    except ImportError:
        print(f'Installing {pip_name}…')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pip_name, '-q'])

for pkg, pip in [('httpx', None), ('pandas', None), ('spacy', None),
                 ('trafilatura', None), ('lxml_html_clean', None)]:
    ensure(pkg, pip)

import csv, json, re, time, urllib.robotparser
from itertools import combinations
from pathlib import Path
from urllib.parse import urljoin, urlparse

import httpx
import pandas as pd
import spacy
import trafilatura

print('Python:', sys.executable)
print('Libraries loaded.')

In [ ]:
# ── 1. Configuration ───────────────────────────────────────────────────────
def find_root(start: Path) -> Path:
    for p in [start.resolve()] + list(start.resolve().parents):
        if (p / 'td1').exists():
            return p
    raise FileNotFoundError('Cannot find workspace root (look for td1 folder)')

ROOT       = find_root(Path.cwd())
TD1        = ROOT / 'td1'
OUTPUT_DIR = TD1 / 'outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_JSONL     = OUTPUT_DIR / 'crawler_output.jsonl'
OUT_ENTITIES  = OUTPUT_DIR / 'extracted_knowledge.csv'
OUT_RELATIONS = OUTPUT_DIR / 'relation_candidates.csv'
OUT_SUMMARY   = OUTPUT_DIR / 'crawl_summary.csv'

# 12 seed URLs — all Wikipedia renewable-energy articles (text-heavy, public)
SEED_URLS = [
    'https://en.wikipedia.org/wiki/Renewable_energy',
    'https://en.wikipedia.org/wiki/Solar_power',
    'https://en.wikipedia.org/wiki/Wind_power',
    'https://en.wikipedia.org/wiki/Hydroelectricity',
    'https://en.wikipedia.org/wiki/Geothermal_energy',
    'https://en.wikipedia.org/wiki/Bioenergy',
    'https://en.wikipedia.org/wiki/Photovoltaics',
    'https://en.wikipedia.org/wiki/Energy_storage',
    'https://en.wikipedia.org/wiki/Sustainable_energy',
    'https://en.wikipedia.org/wiki/International_Renewable_Energy_Agency',
    'https://en.wikipedia.org/wiki/Intergovernmental_Panel_on_Climate_Change',
    'https://en.wikipedia.org/wiki/International_Energy_Agency',
]

MIN_WORDS       = 500   # Lab requirement: > 500 words
REQUEST_TIMEOUT = 30

print(f'Root        : {ROOT}')
print(f'Output dir  : {OUTPUT_DIR}')
print(f'Seed URLs   : {len(SEED_URLS)}')
print(f'Min words   : {MIN_WORDS}')

In [ ]:
# ── 2. robots.txt helper ───────────────────────────────────────────────────
# The lab explicitly requires respecting web ethics / robots.txt
_robots_cache: dict[str, urllib.robotparser.RobotFileParser] = {}

def can_fetch(url: str, user_agent: str = 'TD1-Bot') -> bool:
    """Return True if robots.txt permits fetching *url*."""
    parsed = urlparse(url)
    base   = f'{parsed.scheme}://{parsed.netloc}'
    if base not in _robots_cache:
        rp = urllib.robotparser.RobotFileParser()
        rp.set_url(f'{base}/robots.txt')
        try:
            rp.read()
        except Exception:
            pass  # if robots.txt unreachable, allow by default
        _robots_cache[base] = rp
    return _robots_cache[base].can_fetch(user_agent, url)

# Quick test
for u in SEED_URLS[:2]:
    print(f'  can_fetch({u}) = {can_fetch(u)}')

In [ ]:
# ── 3. Crawl & clean ───────────────────────────────────────────────────────
def normalize(text: str) -> str:
    return re.sub(r'\s+', ' ', text).strip()

def fetch_page_title(html: str) -> str:
    """Extract <title> tag text from raw HTML."""
    m = re.search(r'<title[^>]*>([^<]+)</title>', html, re.IGNORECASE)
    return m.group(1).strip() if m else ''

def is_useful(text: str) -> bool:
    """Return True if the extracted text has > MIN_WORDS words."""
    return len(text.split()) > MIN_WORDS

records = []

for url in SEED_URLS:
    status     = 'ok'
    text       = ''
    title      = ''
    word_count = 0

    # ── robots.txt check ──
    if not can_fetch(url):
        records.append({'url': url, 'title': '', 'status': 'blocked_by_robots', 'word_count': 0, 'text': ''})
        print(f'BLOCKED  {url}')
        continue

    try:
        # trafilatura handles fetch + extraction in one step
        downloaded = trafilatura.fetch_url(url)
        if not downloaded:
            status = 'fetch_failed'
        else:
            title = fetch_page_title(downloaded)
            extracted = trafilatura.extract(
                downloaded,
                include_comments=False,
                include_tables=False,
                favor_precision=True,
            )
            if not extracted:
                status = 'empty_extract'
            else:
                text       = normalize(extracted)
                word_count = len(text.split())
                if not is_useful(text):
                    status = 'too_short'
    except Exception as exc:
        status = f'error: {exc}'

    records.append({'url': url, 'title': title, 'status': status, 'word_count': word_count,
                    'text': text if status == 'ok' else ''})
    print(f'{status:15s}  {word_count:6d} words  {url}')

df_crawl = pd.DataFrame(records)
df_ok    = df_crawl[df_crawl['status'] == 'ok'].copy()

# ── Save JSONL ──
with OUT_JSONL.open('w', encoding='utf-8') as f:
    for _, row in df_ok.iterrows():
        f.write(json.dumps({'url': row['url'], 'title': row['title'],
                            'word_count': int(row['word_count']), 'text': row['text']},
                           ensure_ascii=False) + '\n')

df_crawl.to_csv(OUT_SUMMARY, index=False)

print(f'\nCrawled: {len(df_crawl)}  |  Useful (> {MIN_WORDS} words): {len(df_ok)}')
print(f'JSONL → {OUT_JSONL}')
df_crawl[['url', 'status', 'word_count']]

In [ ]:
# ── 4. Load NLP model (prefer en_core_web_trf as required by lab) ──────────
def load_nlp():
    for model in ['en_core_web_trf', 'en_core_web_lg', 'en_core_web_md', 'en_core_web_sm']:
        try:
            m = spacy.load(model)
            print(f'Loaded spaCy model: {model}')
            return m
        except OSError:
            continue
    # Last resort: download sm
    print('No spaCy model found, downloading en_core_web_sm …')
    subprocess.check_call([sys.executable, '-m', 'spacy', 'download', 'en_core_web_sm', '-q'])
    m = spacy.load('en_core_web_sm')
    print('Loaded: en_core_web_sm (fallback)')
    return m

nlp = load_nlp()

# Confirm model components
print('Components:', nlp.pipe_names)

In [ ]:
# ── 5. NER — extract PERSON, ORG, GPE, DATE ────────────────────────────────
# Noise filters: single chars, unit abbreviations, boilerplate strings
VALID_LABELS = {'PERSON', 'ORG', 'GPE', 'DATE'}

# Stop-list for entities that are clearly wrong (technical abbreviations etc.)
STOP_ENTITIES = {
    'twh', 'gwh', 'mwh', 'kwh', 'gw', 'mw', 'kw', 'pv', 'csp', 'ac', 'dc',
    'co2', 'co₂', 'ghg', 'un', 'eu', 'us', 'uk', 'usd', 'eur',
}

def is_valid_entity(text: str, label: str) -> bool:
    t = text.strip()
    if len(t) < 2:
        return False
    if t.lower() in STOP_ENTITIES:
        return False
    if label in ('PERSON', 'ORG', 'GPE') and re.fullmatch(r'[A-Z]{1,4}', t):
        # Short all-caps abbreviation that is not an org we want
        return False
    if not re.search(r'[A-Za-z]', t):
        return False
    return True

entity_rows = []

for _, row in df_ok.iterrows():
    url = row['url']
    # Process in 100k-char chunks to avoid memory issues with large texts
    text = row['text']
    doc  = nlp(text[:100_000])

    seen = set()
    for ent in doc.ents:
        if ent.label_ not in VALID_LABELS:
            continue
        value = ent.text.strip()
        if not is_valid_entity(value, ent.label_):
            continue
        key = (value.lower(), ent.label_, url)
        if key in seen:
            continue
        seen.add(key)
        entity_rows.append({'Entity_Name': value, 'Type': ent.label_, 'Source_URL': url})

df_entities = pd.DataFrame(entity_rows).drop_duplicates()
df_entities.to_csv(OUT_ENTITIES, index=False)

print(f'Entity rows  : {len(df_entities)}')
print(f'Entity types : {df_entities["Type"].value_counts().to_dict()}')
print(f'Entity CSV   : {OUT_ENTITIES}')
df_entities.head(20)

In [ ]:
# ── 6. Relation extraction via dependency parsing (nsubj → ROOT verb → dobj) 
#       As required by the lab: find the verb connecting two entities
# ─────────────────────────────────────────────────────────────────────────────
relation_rows = []

for _, row in df_ok.iterrows():
    url = row['url']
    doc = nlp(row['text'][:100_000])

    for sent in doc.sents:
        # Collect named entities in this sentence
        sent_ents = [e for e in sent.ents if e.label_ in VALID_LABELS
                     and is_valid_entity(e.text.strip(), e.label_)]
        if len(sent_ents) < 2:
            continue

        # Build a token → entity lookup
        tok_to_ent: dict[int, spacy.tokens.Span] = {}
        for ent in sent_ents:
            for tok in ent:
                tok_to_ent[tok.i] = ent

        # Strategy 1: nsubj → ROOT(VERB) → dobj/attr  (direct triple)
        root = next((t for t in sent if t.dep_ == 'ROOT' and t.pos_ == 'VERB'), None)
        if root:
            subjects = [t for t in root.children if t.dep_ in ('nsubj', 'nsubjpass')]
            objects  = [t for t in root.children if t.dep_ in ('dobj', 'attr', 'pobj', 'obj')]

            for subj_tok in subjects:
                subj_ent = tok_to_ent.get(subj_tok.i)
                if subj_ent is None:
                    # Try the head of the subject NP
                    for child in subj_tok.subtree:
                        if child.i in tok_to_ent:
                            subj_ent = tok_to_ent[child.i]
                            break
                if subj_ent is None:
                    continue

                for obj_tok in objects:
                    obj_ent = tok_to_ent.get(obj_tok.i)
                    if obj_ent is None:
                        for child in obj_tok.subtree:
                            if child.i in tok_to_ent:
                                obj_ent = tok_to_ent[child.i]
                                break
                    if obj_ent is None or obj_ent == subj_ent:
                        continue

                    src  = subj_ent.text.strip()
                    rel  = root.lemma_.lower()
                    tgt  = obj_ent.text.strip()
                    if src.lower() == tgt.lower() or not rel:
                        continue

                    relation_rows.append({
                        'subject':   src,
                        'predicate': rel,
                        'object':    tgt,
                        'subject_type': subj_ent.label_,
                        'object_type':  obj_ent.label_,
                        'sentence':  sent.text.strip()[:300],
                        'source_url': url,
                    })

df_rel = pd.DataFrame(relation_rows)
if not df_rel.empty:
    df_rel = df_rel.drop_duplicates(subset=['subject', 'predicate', 'object', 'source_url'])

df_rel.to_csv(OUT_RELATIONS, index=False)

print(f'Relation candidates : {len(df_rel)}')
print(f'Relations CSV       : {OUT_RELATIONS}')
df_rel.head(20)

In [ ]:
# ── 7. Quality summary & readiness check for TD4 ──────────────────────────
print('=' * 60)
print('TD1 PIPELINE SUMMARY')
print('=' * 60)
print(f'  Pages crawled          : {len(df_crawl)}')
print(f'  Pages useful (>{MIN_WORDS}w) : {len(df_ok)}')
print(f'  Total entities         : {len(df_entities)}')
print(f'  Entity type breakdown  : {df_entities["Type"].value_counts().to_dict()}')
print(f'  Relation triples       : {len(df_rel)}')
print()
print('Outputs:')
print(f'  {OUT_JSONL}')
print(f'  {OUT_ENTITIES}')
print(f'  {OUT_RELATIONS}')
print(f'  {OUT_SUMMARY}')
print()
if len(df_ok) >= 5 and len(df_entities) >= 100 and len(df_rel) >= 50:
    print('✅ Data is ready for TD4 (KB Construction)')
else:
    print('⚠️  Add more seed URLs to meet minimum data thresholds.')

---
## Lab Report Notes (for 2-page written report)

### 1. Methodology
The pipeline follows three steps:
1. **Crawl** — `trafilatura.fetch_url()` downloads the raw HTML. Before any request, `urllib.robotparser` checks `robots.txt` for the domain; blocked URLs are skipped.
2. **Clean** — `trafilatura.extract()` with `favor_precision=True` isolates the main article body from navigation, footers, and ads. A word-count gate (> 500 words) filters thin pages.
3. **Extract** — spaCy's `en_core_web_trf` (transformer-based) tags PERSON, ORG, GPE, DATE entities; a stop-list drops unit abbreviations (GW, TWh, etc.). Dependency parsing (nsubj→ROOT-VERB→dobj) extracts S-P-O triples.

### 2. Entity Ambiguity Examples
| Entity string | Predicted label | Correct label | Reason for error |
|---|---|---|---|
| COVID-19 | PERSON | none (not an entity) | Named-entity model over-generalizes proper nouns |
| Golmud Solar Park | PERSON | FAC (facility) | Park names sometimes resemble person names |
| G7 | GPE | ORG | Model treats acronym as a geo-political entity |

### 3. Scaling Reflection
For 10,000 pages:
- **Async crawling** (`asyncio` + `httpx.AsyncClient`) with rate limiting per domain.
- **Distributed queue** (Redis / Celery) to manage crawl frontiers.
- **Batch NLP** (`nlp.pipe()`) with GPU to process text in parallel.
- **Deduplication** via URL normalization and content hashing (SimHash).
- **Distributed RDF store** (Apache Jena / Blazegraph) to store and query the resulting KG.